# ==========================================
# WBET Streamflow Retrieval
#
# Step 1
# ==========================================

# Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Step 2: Install packages

In [ ]:
!pip install dataretrieval pandas geopandas tqdm openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.8/164.8 kB 1.4 MB/s eta 0:00:00


# Step 3: Import the input file.

In [ ]:
import pandas as pd
from pathlib import Path

input_dir = Path("/content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval/data/input")

gauge_table = pd.read_csv(
    input_dir / "gauge_matches_selected_basins.csv",
    dtype={"HUCID": str, "gauge_in": str, "gauge_out": str}
)

gauge_table = gauge_table.rename(columns={"HUCID": "huc8"})

gauge_table.head()

,huc8,gauge_in,gauge_out
0,18100204,USGS-09527700,USGS-10259540
1,18090202,NaN,USGS-10251330
2,18070203,NaN,USGS-11078000
3,18070202,NaN,USGS-11070500
4,18060005,NaN,USGS-11152500


# Step 4: Standardize required fields and convert to long format file

In [ ]:
# Standardize required fields
required_cols = ["huc8", "gauge_in", "gauge_out"]

missing_cols = [col for col in required_cols if col not in gauge_table.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

gauge_table["huc8"] = gauge_table["huc8"].astype(str)


# Helper to clean USGS gauge IDs
def clean_site_no(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    x = x.replace("USGS-", "")
    return x if x != "" else None


# Convert from wide format to long format
gauge_long = gauge_table.melt(
    id_vars=["huc8"],
    value_vars=["gauge_in", "gauge_out"],
    var_name="role",
    value_name="site_no"
)

gauge_long["role"] = gauge_long["role"].replace({
    "gauge_in": "inflow",
    "gauge_out": "outflow"
})

gauge_long["site_no"] = gauge_long["site_no"].apply(clean_site_no)

gauge_long = (
    gauge_long
    .dropna(subset=["site_no"])
    .query("site_no != ''")
    .drop_duplicates()
    .reset_index(drop=True)
)

gauge_long.head()

,huc8,role,site_no
0,18100204,inflow,09527700
1,17040212,inflow,13087995
2,17040209,inflow,13077000
3,18100204,outflow,10259540
4,18090202,outflow,10251330


# Step 5: Build a table containing one record for each unique USGS gauge for data retrieval


In [ ]:
unique_gauges = (
    gauge_long
    .groupby("site_no")
    .agg(
        huc8=("huc8", "first"),
        role=("role", "first"),
        n_huc_roles=("huc8", "count"),
        all_huc_roles=("role", lambda x: ";".join(
            gauge_long.loc[x.index, "huc8"] + "_" + x
        ))
    )
    .reset_index()
    .sort_values("site_no")
)

print(f"{len(unique_gauges)} unique gauges")

unique_gauges.head()

114 unique gauges


,site_no,huc8,role,n_huc_roles,all_huc_roles
0,09527700,18100204,inflow,1,18100204_inflow
1,10251330,18090202,outflow,1,18090202_outflow
2,10259540,18100204,outflow,1,18100204_outflow
3,11070500,18070202,outflow,1,18070202_outflow
4,11078000,18070203,outflow,1,18070203_outflow


# Step 6: Set-up directory structures

In [ ]:
from pathlib import Path
import pandas as pd
import time

# Project folders
project_dir = Path("/content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval")

raw_dir = project_dir / "data" / "streamflow" / "raw"
monthly_dir = project_dir / "data" / "streamflow" / "monthly"
reports_dir = project_dir / "data" / "reports"

for folder in [raw_dir, monthly_dir, reports_dir]:
    folder.mkdir(parents=True, exist_ok=True)

# Analysis settings
start_date = "2000-01-01"
end_date = "2025-12-31"
parameter_code = "00060"  # USGS daily discharge


def make_usgs_id(site_no):
    """Convert a site number to USGS monitoring-location format."""
    site_no = str(site_no).strip()
    if site_no.startswith("USGS-"):
        return site_no
    return f"USGS-{site_no}"


def make_raw_filename(site, huc, role, start_date, end_date):
    """Create standardized raw streamflow filename."""
    clean_start = start_date.replace("-", "")
    clean_end = end_date.replace("-", "")
    return f"USGS_{site}_HUC{huc}_{role}_{clean_start}_{clean_end}.csv"


def read_existing_streamflow_csv(file):
    """Read an existing streamflow CSV as text to avoid type conflicts."""
    return pd.read_csv(file, dtype=str)


print("Folders created and helper functions loaded.")

Folders created and helper functions loaded.


# Step 7: TEST on just one USGS download in Python before looping all gauges.

In [ ]:
import dataretrieval.waterdata as wd

test_site = unique_gauges.iloc[0]["site_no"]
test_site_id = make_usgs_id(test_site)

print("Testing:", test_site_id)

test_daily, test_meta = wd.get_daily(   # wd.get_daily() returns two things: the data table and metadata about the API request.
    monitoring_location_id=test_site_id,
    parameter_code=parameter_code,
    time=[start_date, "2020-01-31"]
)

print(test_daily.shape)
test_daily.head()

Testing: USGS-09527700
(3020, 12)


,geometry,time_series_id,monitoring_location_id,parameter_code,statistic_id,time,value,unit_of_measure,approval_status,qualifier,last_modified,daily_id
0,POINT (-115.04811 32.70425),e197b96685c8425580976c391d2b55f3,USGS-09527700,00060,00003,2011-10-26,3600,ft^3/s,Approved,None,2025-03-10 20:38:50.184973+00:00,40b591a0-cf8c-4870-bb42-907ebedf7e20
1,POINT (-115.04811 32.70425),e197b96685c8425580976c391d2b55f3,USGS-09527700,00060,00003,2011-10-27,3730,ft^3/s,Approved,None,2025-03-10 20:38:50.184973+00:00,9fd08ef0-5b13-46fc-a411-70b926ff7cc9
2,POINT (-115.04811 32.70425),e197b96685c8425580976c391d2b55f3,USGS-09527700,00060,00003,2011-10-28,3520,ft^3/s,Approved,None,2025-03-10 20:38:50.184973+00:00,ef08900e-0f5a-4c39-9a86-02f7fefb4247
3,POINT (-115.04811 32.70425),e197b96685c8425580976c391d2b55f3,USGS-09527700,00060,00003,2011-10-29,3150,ft^3/s,Approved,None,2025-03-10 20:38:50.184973+00:00,3c323901-f178-4669-9d91-ec6c3e292117
4,POINT (-115.04811 32.70425),e197b96685c8425580976c391d2b55f3,USGS-09527700,00060,00003,2011-10-30,3010,ft^3/s,Approved,None,2025-03-10 20:38:50.184973+00:00,c0fe2426-f216-43d7-95d8-d557291bedf0


# Step 8: Python equivalent of safe_read_daily() function
download daily stremflow one gauge at a time, return a pandas dataframe. If download fails create 1-row dataframe describing error.

In [ ]:
import pandas as pd
import dataretrieval.waterdata as wd

def safe_read_daily(site_no,
                    start_date,
                    end_date,
                    parameter_code="00060"):
    """
    Download daily streamflow for one USGS gauge.

    Returns a pandas DataFrame.

    If the download fails, a one-row DataFrame describing
    the error is returned instead.
    """

    site_id = make_usgs_id(site_no)

    try:

        dat, meta = wd.get_daily(
            monitoring_location_id=site_id,
            parameter_code=parameter_code,
            time=[start_date, end_date]
        )

        dat = dat.copy()

        dat["site_no"] = str(site_no)
        dat["download_status"] = "ok"

        return dat

    except Exception as e:

        return pd.DataFrame([{
            "site_no": str(site_no),
            "download_status": "failed",
            "retrieval_error": str(e)
        }])

# Restartable loop. This should skip existing raw files and download missing ones.

In [ ]:
from tqdm.auto import tqdm

download_log = []

for i, row in tqdm(unique_gauges.iterrows(), total=len(unique_gauges)):

    site = row["site_no"]
    huc = row["huc8"]
    role = row["role"]

    out_file = raw_dir / make_raw_filename(
        site=site,
        huc=huc,
        role=role,
        start_date=start_date,
        end_date=end_date
    )

    # Restart logic: skip any gauge file that already exists.
    if out_file.exists():

        existing_dat = pd.read_csv(out_file, dtype=str)

        file_status = (
            existing_dat["download_status"].iloc[0]
            if "download_status" in existing_dat.columns and len(existing_dat) > 0
            else "existing_no_data"
        )

        download_log.append({
            "site_no": site,
            "huc8": huc,
            "role": role,
            "n_huc_roles": row["n_huc_roles"],
            "all_huc_roles": row["all_huc_roles"],
            "download_status": file_status,
            "rows": len(existing_dat),
            "elapsed_seconds": 0,
            "file": str(out_file)
        })

        print(f"{i+1} of {len(unique_gauges)} - Skipping {site} - file already exists")
        continue

    print(f"{i+1} of {len(unique_gauges)} - Downloading daily streamflow for {site}")

    t0 = time.time()

    dat = safe_read_daily(
        site_no=site,
        start_date=start_date,
        end_date=end_date,
        parameter_code=parameter_code
    )

    elapsed = round(time.time() - t0, 1)

    if "download_status" in dat.columns and len(dat) > 0:
        download_status = dat["download_status"].iloc[0]
    elif len(dat) == 0:
        download_status = "no_data"
    else:
        download_status = "ok"

    dat.to_csv(out_file, index=False)

    download_log.append({
        "site_no": site,
        "huc8": huc,
        "role": role,
        "n_huc_roles": row["n_huc_roles"],
        "all_huc_roles": row["all_huc_roles"],
        "download_status": download_status,
        "rows": len(dat),
        "elapsed_seconds": elapsed,
        "file": str(out_file)
    })

    print(
        f"Finished {site} in {elapsed} seconds; "
        f"{len(dat)} rows; {download_status}"
    )

    time.sleep(0.2)

download_log = pd.DataFrame(download_log)

download_log.to_csv(
    reports_dir / "streamflow_download_log.csv",
    index=False
)

download_log.head()

  0%|          | 0/114 [00:00<?, ?it/s]

1 of 114 - Skipping 09527700 - file already exists
2 of 114 - Skipping 10251330 - file already exists
3 of 114 - Skipping 10259540 - file already exists
4 of 114 - Skipping 11070500 - file already exists
5 of 114 - Skipping 11078000 - file already exists
6 of 114 - Skipping 11148500 - file already exists
7 of 114 - Skipping 11152500 - file already exists
8 of 114 - Skipping 11251000 - file already exists
9 of 114 - Skipping 11255575 - file already exists
10 of 114 - Skipping 11303500 - file already exists
11 of 114 - Skipping 11374000 - file already exists
12 of 114 - Skipping 11376000 - file already exists
13 of 114 - Skipping 11376550 - file already exists
14 of 114 - Skipping 11421000 - file already exists
15 of 114 - Skipping 11454000 - file already exists
16 of 114 - Skipping 11473900 - file already exists
17 of 114 - Skipping 11501000 - file already exists
18 of 114 - Skipping 11502500 - file already exists
19 of 114 - Skipping 11509390 - file already exists
20 of 114 - Skipping 

,site_no,huc8,role,n_huc_roles,all_huc_roles,download_status,rows,elapsed_seconds,file
0,09527700,18100204,inflow,1,18100204_inflow,ok,5181,0,/content/drive/MyDrive/CSUMB/NASA/Projects/Run...
1,10251330,18090202,outflow,1,18090202_outflow,ok,7142,0,/content/drive/MyDrive/CSUMB/NASA/Projects/Run...
2,10259540,18100204,outflow,1,18100204_outflow,ok,9443,0,/content/drive/MyDrive/CSUMB/NASA/Projects/Run...
3,11070500,18070202,outflow,1,18070202_outflow,ok,9497,0,/content/drive/MyDrive/CSUMB/NASA/Projects/Run...
4,11078000,18070203,outflow,1,18070203_outflow,ok,9497,0,/content/drive/MyDrive/CSUMB/NASA/Projects/Run...


# Generate Combined Daily Dataset

In [ ]:
# Combine all raw gauge CSVs into one daily dataset

raw_files = sorted(raw_dir.glob("*.csv"))

daily_raw = pd.concat(
    [pd.read_csv(f, dtype=str) for f in raw_files],
    ignore_index=True
)

daily_raw.to_csv(
    project_dir / "data" / "streamflow" / "daily_streamflow_all_gauges.csv",
    index=False
)

print(daily_raw.shape)
daily_raw.head()

(924343, 15)


,geometry,time_series_id,monitoring_location_id,parameter_code,statistic_id,time,value,unit_of_measure,approval_status,qualifier,last_modified,daily_id,site_no,download_status,status
0,POINT (-115.048113888889 32.70425),e197b96685c8425580976c391d2b55f3,USGS-09527700,00060,00003,2011-10-26,3600,ft^3/s,Approved,NaN,2025-03-10 20:38:50.184973+00:00,40b591a0-cf8c-4870-bb42-907ebedf7e20,09527700,ok,NaN
1,POINT (-115.048113888889 32.70425),e197b96685c8425580976c391d2b55f3,USGS-09527700,00060,00003,2011-10-27,3730,ft^3/s,Approved,NaN,2025-03-10 20:38:50.184973+00:00,9fd08ef0-5b13-46fc-a411-70b926ff7cc9,09527700,ok,NaN
2,POINT (-115.048113888889 32.70425),e197b96685c8425580976c391d2b55f3,USGS-09527700,00060,00003,2011-10-28,3520,ft^3/s,Approved,NaN,2025-03-10 20:38:50.184973+00:00,ef08900e-0f5a-4c39-9a86-02f7fefb4247,09527700,ok,NaN
3,POINT (-115.048113888889 32.70425),e197b96685c8425580976c391d2b55f3,USGS-09527700,00060,00003,2011-10-29,3150,ft^3/s,Approved,NaN,2025-03-10 20:38:50.184973+00:00,3c323901-f178-4669-9d91-ec6c3e292117,09527700,ok,NaN
4,POINT (-115.048113888889 32.70425),e197b96685c8425580976c391d2b55f3,USGS-09527700,00060,00003,2011-10-30,3010,ft^3/s,Approved,NaN,2025-03-10 20:38:50.184973+00:00,c0fe2426-f216-43d7-95d8-d557291bedf0,09527700,ok,NaN


In [ ]:
# Standardize fields for quality reporting

daily_streamflow = daily_raw.copy()

daily_streamflow["time"] = pd.to_datetime(daily_streamflow["time"], errors="coerce")
daily_streamflow["value"] = pd.to_numeric(daily_streamflow["value"], errors="coerce")

daily_streamflow = daily_streamflow.dropna(subset=["time"])

expected_dates = pd.date_range(start=start_date, end=end_date, freq="D")
n_days_expected = len(expected_dates)

daily_quality = (
    daily_streamflow
    .groupby("site_no")
    .agg(
        requested_start=("time", lambda x: pd.to_datetime(start_date)),
        requested_end=("time", lambda x: pd.to_datetime(end_date)),
        actual_start=("time", "min"),
        actual_end=("time", "max"),
        n_days_returned=("time", "nunique"),
        n_duplicate_dates=("time", lambda x: len(x) - x.nunique()),
        n_missing_values=("value", lambda x: x.isna().sum())
    )
    .reset_index()
)

daily_quality["n_days_expected"] = n_days_expected
daily_quality["n_missing_days"] = (
    daily_quality["n_days_expected"] - daily_quality["n_days_returned"]
)

daily_quality["percent_complete"] = (
    100 * daily_quality["n_days_returned"] / daily_quality["n_days_expected"]
).round(1)

daily_quality["quality_status"] = pd.cut(
    daily_quality["percent_complete"],
    bins=[-1, 0, 75, 95, 100],
    labels=["no_data", "limited_record", "partial_record", "complete_or_nearly_complete"]
)

all_quality = (
    unique_gauges
    .merge(daily_quality, on="site_no", how="left")
    .merge(download_log[["site_no", "download_status", "rows", "elapsed_seconds", "file"]],
           on="site_no", how="left")
)

all_quality["n_days_returned"] = all_quality["n_days_returned"].fillna(0).astype(int)
all_quality["n_days_expected"] = all_quality["n_days_expected"].fillna(n_days_expected).astype(int)
all_quality["n_missing_days"] = all_quality["n_missing_days"].fillna(n_days_expected).astype(int)
all_quality["percent_complete"] = all_quality["percent_complete"].fillna(0)

all_quality["quality_status"] = all_quality["quality_status"].astype(str).replace("nan", "no_data")

all_quality.to_csv(
    reports_dir / "gauge_daily_quality_report.csv",
    index=False
)

all_quality.head()

,site_no,huc8,role,n_huc_roles,all_huc_roles,requested_start,requested_end,actual_start,actual_end,n_days_returned,n_duplicate_dates,n_missing_values,n_days_expected,n_missing_days,percent_complete,quality_status,download_status,rows,elapsed_seconds,file
0,09527700,18100204,inflow,1,18100204_inflow,2000-01-01,2019-12-31,2011-10-26,2025-12-31,5181,0.0,0.0,7305,2124,70.9,limited_record,ok,2989,1.4,/content/drive/MyDrive/CSUMB/NASA/Projects/Run...
1,10251330,18090202,outflow,1,18090202_outflow,2000-01-01,2019-12-31,2006-06-13,2025-12-31,7142,0.0,0.0,7305,163,97.8,complete_or_nearly_complete,ok,4950,2.2,/content/drive/MyDrive/CSUMB/NASA/Projects/Run...
2,10259540,18100204,outflow,1,18100204_outflow,2000-01-01,2019-12-31,2000-01-01,2025-12-31,9443,0.0,0.0,7305,-2138,129.3,no_data,ok,7262,2.3,/content/drive/MyDrive/CSUMB/NASA/Projects/Run...
3,11070500,18070202,outflow,1,18070202_outflow,2000-01-01,2019-12-31,2000-01-01,2025-12-31,9497,0.0,0.0,7305,-2192,130.0,no_data,ok,7305,2.3,/content/drive/MyDrive/CSUMB/NASA/Projects/Run...
4,11078000,18070203,outflow,1,18070203_outflow,2000-01-01,2019-12-31,2000-01-01,2025-12-31,9497,0.0,0.0,7305,-2192,130.0,no_data,ok,7305,2.1,/content/drive/MyDrive/CSUMB/NASA/Projects/Run...


HUC8 Role Quality Report

In [ ]:
huc_role_quality = gauge_long.merge(
    all_quality[
        [
            "site_no",
            "requested_start",
            "requested_end",
            "actual_start",
            "actual_end",
            "n_days_expected",
            "n_days_returned",
            "n_missing_days",
            "percent_complete",
            "download_status",
            "quality_status"
        ]
    ],
    on="site_no",
    how="left"
).sort_values(["huc8", "role", "site_no"])

huc_role_quality.to_csv(
    reports_dir / "huc8_gauge_role_quality_report.csv",
    index=False
)

huc_role_quality.head()

,huc8,role,site_no,requested_start,requested_end,actual_start,actual_end,n_days_expected,n_days_returned,n_missing_days,percent_complete,download_status,quality_status
119,17010102,outflow,12302055,2000-01-01,2019-12-31,2000-01-01,2025-12-31,7305,9497,-2192,130.0,ok,no_data
118,17010103,outflow,12304500,2000-01-01,2019-12-31,2000-01-01,2025-12-31,7305,9497,-2192,130.0,ok,no_data
117,17010201,outflow,12324680,2000-01-01,2019-12-31,2000-01-01,2025-12-31,7305,9497,-2192,130.0,ok,no_data
116,17010202,outflow,12334550,2000-01-01,2019-12-31,2000-01-01,2025-12-31,7305,9497,-2192,130.0,ok,no_data
115,17010203,outflow,12340000,2000-01-01,2019-12-31,2000-01-01,2025-12-31,7305,9497,-2192,130.0,ok,no_data


# Create Monthly stremflow summary

In [ ]:
# Monthly streamflow summaries

monthly_streamflow = daily_streamflow.copy()

monthly_streamflow["year"] = monthly_streamflow["time"].dt.year
monthly_streamflow["month"] = monthly_streamflow["time"].dt.month
monthly_streamflow["water_year"] = monthly_streamflow["year"]

monthly_streamflow.loc[
    monthly_streamflow["month"] >= 10,
    "water_year"
] += 1

monthly_streamflow["year_month"] = monthly_streamflow["time"].dt.to_period("M").dt.to_timestamp()

monthly_streamflow = (
    monthly_streamflow
    .groupby(["site_no", "year_month", "water_year", "year", "month"])
    .agg(
        mean_discharge_cfs=("value", "mean"),
        sum_daily_mean_cfs=("value", "sum"),
        n_days=("time", "nunique"),
        n_missing_values=("value", lambda x: x.isna().sum())
    )
    .reset_index()
)

monthly_streamflow.to_csv(
    monthly_dir / "all_gauges_monthly_streamflow_summary.csv",
    index=False
)

monthly_streamflow.head()

,site_no,year_month,water_year,year,month,mean_discharge_cfs,sum_daily_mean_cfs,n_days,n_missing_values
0,09527700,2011-10-01,2012,2011,10,3350.000000,20100.0,6,0
1,09527700,2011-11-01,2012,2011,11,2551.000000,76530.0,30,0
2,09527700,2011-12-01,2012,2011,12,1904.451613,59038.0,31,0
3,09527700,2012-01-01,2012,2012,1,2339.548387,72526.0,31,0
4,09527700,2012-02-01,2012,2012,2,2935.862069,85140.0,29,0


# Export 1-monthly csv for each month.

In [ ]:
for site in unique_gauges["site_no"]:
    this_data = monthly_streamflow[monthly_streamflow["site_no"] == site]

    out_file = monthly_dir / f"USGS_{site}_monthly_{start_date.replace('-', '')}_{end_date.replace('-', '')}.csv"

    this_data.to_csv(out_file, index=False)

print("Monthly gauge files exported.")

Monthly gauge files exported.


# make table that has the year across the column and the stations in a column called station. the cells in each year would be the percentage of the water year that we have data for from a station.
which gauges have full water years
which gauges begin late
which gauges terminate early
which gauges are unusable

This is exactly the table I'd want before doing WBET.

In [ ]:
import pandas as pd

#-------------------------------------------------------
# Assign Water Year
#-------------------------------------------------------

wy = daily_streamflow.copy()

wy["water_year"] = wy["time"].dt.year

wy.loc[
    wy["time"].dt.month >= 10,
    "water_year"
] += 1

#-------------------------------------------------------
# Count observations in each water year
#-------------------------------------------------------

wy_summary = (
    wy
    .groupby(["site_no","water_year"])
    .agg(
        n_days=("time","nunique")
    )
    .reset_index()
)

#-------------------------------------------------------
# Determine expected number of days in each water year
#-------------------------------------------------------

def water_year_days(wy):

    start = pd.Timestamp(year=wy-1, month=10, day=1)
    end   = pd.Timestamp(year=wy, month=9, day=30)

    return len(pd.date_range(start, end))

wy_summary["expected_days"] = wy_summary["water_year"].apply(water_year_days)

wy_summary["percent_complete"] = (
    100
    * wy_summary["n_days"]
    / wy_summary["expected_days"]
).round(1)

#-------------------------------------------------------
# Pivot into a matrix
#-------------------------------------------------------

wy_matrix = (
    wy_summary
    .pivot(
        index="site_no",
        columns="water_year",
        values="percent_complete"
    )
    .fillna(0)
)

wy_matrix.style.background_gradient(
    cmap="RdYlGn",
    axis=None
)

water_year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
site_no,,,,,,,,,,,,,,,,,,,,,,,,,,,
09527700,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,93.200000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,25.200000
10251330,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,30.100000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,25.200000
10259540,74.900000,100.000000,100.000000,100.000000,100.000000,97.500000,100.000000,100.000000,100.000000,100.000000,97.800000,98.100000,99.700000,99.500000,99.200000,99.500000,98.600000,98.900000,100.000000,99.500000,99.500000,100.000000,100.000000,97.500000,100.000000,100.000000,25.200000
11070500,74.900000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,25.200000
11078000,74.900000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,25.200000
11148500,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,100.000000,100.000000,100.000000,34.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
11152500,74.900000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,24.900000
11251000,74.900000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,25.200000
11255575,49.700000,58.100000,58.100000,58.100000,58.200000,58.100000,58.100000,58.100000,58.200000,58.100000,58.100000,58.100000,58.200000,58.100000,58.100000,58.100000,58.200000,58.100000,58.100000,58.100000,58.200000,58.100000,57.800000,96.200000,66.400000,49.900000,8.500000


In [ ]:
wy_matrix.to_csv(
    reports_dir / "water_year_completeness.csv"
)

Assign coordinates to station inventory

In [ ]:
import re

def extract_lon(point_text):
    if pd.isna(point_text):
        return None
    match = re.search(r"POINT \(([-\d.]+) ([-\d.]+)\)", str(point_text))
    return float(match.group(1)) if match else None

def extract_lat(point_text):
    if pd.isna(point_text):
        return None
    match = re.search(r"POINT \(([-\d.]+) ([-\d.]+)\)", str(point_text))
    return float(match.group(2)) if match else None


station_locations = (
    daily_streamflow
    .dropna(subset=["geometry"])
    .groupby("site_no")
    .first()
    .reset_index()
)

station_locations["longitude"] = station_locations["geometry"].apply(extract_lon)
station_locations["latitude"] = station_locations["geometry"].apply(extract_lat)

station_locations = station_locations[
    ["site_no", "latitude", "longitude"]
]

station_locations.head()

,site_no,latitude,longitude
0,09527700,32.704250,-115.048114
1,10251330,35.790528,-116.207639
2,10259540,33.524759,-116.077496
3,11070500,33.664189,-117.293927
4,11078000,33.751128,-117.908391


In [ ]:
# Merge into water-year station inventory
station_inventory = (
    unique_gauges
    .merge(station_locations, on="site_no", how="left")
    .merge(wy_matrix, left_on="site_no", right_index=True, how="left")
)

station_inventory.to_csv(
    reports_dir / "station_inventory.csv",
    index=False
)

station_inventory.head()

,site_no,huc8,role,n_huc_roles,all_huc_roles,latitude,longitude,2020,2021,2022,2023,2024,2025,2026
0,09527700,18100204,inflow,1,18100204_inflow,32.704250,-115.048114,74.9,100.0,100.0,100.0,100.0,100.0,25.2
1,10251330,18090202,outflow,1,18090202_outflow,35.790528,-116.207639,74.9,100.0,100.0,100.0,100.0,100.0,25.2
2,10259540,18100204,outflow,1,18100204_outflow,33.524759,-116.077496,74.3,100.0,100.0,97.5,100.0,100.0,25.2
3,11070500,18070202,outflow,1,18070202_outflow,33.664189,-117.293927,74.9,100.0,100.0,100.0,100.0,100.0,25.2
4,11078000,18070203,outflow,1,18070203_outflow,33.751128,-117.908391,74.9,100.0,100.0,100.0,100.0,100.0,25.2


Map Time!

In [ ]:
!pip install folium

In [ ]:
import folium
import pandas as pd

# Choose a water year to color by.
# You can change this to another year column from wy_matrix.
map_water_year = station_inventory.columns[-2]

def completeness_color(value):
    try:
        value = float(value)
    except:
        return "gray"

    if value >= 95:
        return "green"
    elif value >= 75:
        return "orange"
    elif value > 0:
        return "red"
    else:
        return "gray"

# Keep only stations with coordinates
map_data = station_inventory.dropna(subset=["latitude", "longitude"]).copy()

# Center map on stations
center_lat = map_data["latitude"].astype(float).mean()
center_lon = map_data["longitude"].astype(float).mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=5,
    tiles="OpenStreetMap"
)

for _, row in map_data.iterrows():

    completeness = row.get(map_water_year, "NA")
    color = completeness_color(completeness)

    popup = f"""
    <b>USGS Station:</b> {row['site_no']}<br>
    <b>Representative HUC8:</b> {row['huc8']}<br>
    <b>Role:</b> {row['role']}<br>
    <b>HUC Assignments:</b> {row['n_huc_roles']}<br>
    <b>{map_water_year} Completeness:</b> {completeness}%<br>
    <b>Latitude:</b> {row['latitude']}<br>
    <b>Longitude:</b> {row['longitude']}
    """

    folium.CircleMarker(
        location=[float(row["latitude"]), float(row["longitude"])],
        radius=6,
        popup=folium.Popup(popup, max_width=350),
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8
    ).add_to(m)

# Add legend
legend_html = f"""
<div style="
position: fixed;
bottom: 40px;
left: 40px;
width: 220px;
height: 140px;
background-color: white;
border:2px solid grey;
z-index:9999;
font-size:14px;
padding: 10px;
">
<b>Water Year Completeness</b><br>
Year: {map_water_year}<br>
<span style="color:green;">●</span> ≥95% complete<br>
<span style="color:orange;">●</span> 75–95% complete<br>
<span style="color:red;">●</span> 1–75% complete<br>
<span style="color:gray;">●</span> 0% / no data<br>
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

m

Save the map as an html?

In [ ]:
map_file = reports_dir / f"WBET_Stream_Gauge_Map_{map_water_year}.html"

m.save(map_file)

print(f"Saved map to: {map_file}")

Saved map to: /content/drive/MyDrive/CSUMB/NASA/Projects/Runoff_Delineation_ET/USGS_streamflow_retrieval/data/reports/WBET_Stream_Gauge_Map_2025.html
